# 🔬 Research Assistant — MCP + Gemini + LangGraph

**Theme:** Research Assistant — files + web fetch + custom citation/formatting tools

**Architecture:**
- filesystem MCP server → read/write files
- fetch MCP server → fetch web pages / Wikipedia  
- custom_mcp_server.py → citation formatter + keyword extractor
- **Gemini** (via LangChain) as the LLM brain
- **LangGraph** ReAct agent as the orchestrator (TAO loop)


## 1. Install dependencies

In [ ]:
%pip install -qU \
  "langchain>=0.3" \
  "langgraph>=0.2" \
  "langchain-google-genai>=2.0" \
  "google-genai>=1.0" \
  "langchain-mcp-adapters==0.2.1" \
  "fastmcp>=2.0.0" \
  "nest_asyncio"

print("✅ Dependencies installed")

## 2. Set Google API Key

In [ ]:
import os
from google.colab import userdata

os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
print("GOOGLE_API_KEY set:", bool(os.getenv("GOOGLE_API_KEY")))

## 3. Check Node / NPM

In [ ]:
import subprocess

result = subprocess.run(["node", "--version"], capture_output=True, text=True)
if result.returncode != 0:
    print("Installing Node.js...")
    os.system("apt-get -qq update && apt-get -qq install -y nodejs npm")
else:
    print("Node:", result.stdout.strip())

result2 = subprocess.run(["npx", "--version"], capture_output=True, text=True)
print("NPX:", result2.stdout.strip())

## 4. Setup working directory with sample papers

In [ ]:
import os
from pathlib import Path

WORKDIR = "/content/research_workspace"
Path(WORKDIR).mkdir(parents=True, exist_ok=True)

sample_files = {
    "paper_transformer.txt": """Title: Attention Is All You Need
Authors: Vaswani et al., 2017
Venue: NeurIPS 2017

Abstract:
The Transformer model architecture is based solely on attention mechanisms,
dispensing with recurrence and convolutions entirely. Experiments on two machine
translation tasks show these models are superior in quality while being more
parallelizable. The model achieves 28.4 BLEU on the WMT 2014 English-to-German task.""",
    "paper_bert.txt": """Title: BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding
Authors: Devlin, Jacob, Chang, Ming-Wei, Lee, Kenton, Toutanova, Kristina
Venue: NAACL 2019

Abstract:
BERT stands for Bidirectional Encoder Representations from Transformers. Unlike
recent language representation models, BERT pre-trains deep bidirectional representations
by conditioning on both left and right context in all layers. BERT can be fine-tuned
with one additional output layer to create state-of-the-art models for many NLP tasks.""",
    "paper_gpt3.txt": """Title: Language Models are Few-Shot Learners
Authors: Brown, Tom, Mann, Benjamin, Ryder, Nick
Venue: NeurIPS 2020

Abstract:
GPT-3 is an autoregressive language model with 175 billion parameters. Tested in
the few-shot setting, it achieves strong performance on many NLP datasets. GPT-3
can generate news articles which human evaluators struggle to distinguish from
human-written articles. The model performs well without any gradient updates.""",
}

for filename, content in sample_files.items():
    (Path(WORKDIR) / filename).write_text(content, encoding="utf-8")

print(f"✅ Workspace ready: {WORKDIR}")
print("Files:", [f.name for f in Path(WORKDIR).iterdir()])

## 5. Write custom MCP server

In [ ]:
from pathlib import Path

server_path = Path("/content/custom_mcp_server.py")
server_content = '''
from fastmcp import FastMCP
from typing import Dict, List
import re

mcp = FastMCP(name="research_tools")

@mcp.tool
def ping() -> str:
    """Health check."""
    return "pong"

@mcp.tool
def format_citation(title: str, authors: str, year: int, venue: str) -> str:
    """Format a research paper citation in APA style.
    Args:
        title: Paper title
        authors: Author names comma-separated
        year: Publication year
        venue: Conference or journal name
    """
    author_list = [a.strip() for a in authors.split(",")]
    if len(author_list) > 3:
        author_str = author_list[0] + " et al."
    else:
        author_str = ", ".join(author_list)
    return f"{author_str} ({year}). {title}. {venue}."

@mcp.tool
def extract_keywords(text: str, max_keywords: int = 10) -> Dict[str, object]:
    """Extract keywords and statistics from a research text.
    Args:
        text: The research text to analyze
        max_keywords: Maximum number of keywords to return
    """
    stopwords = {
        "the","a","an","and","or","but","in","on","at","to","for",
        "of","with","by","from","is","are","was","were","be","been",
        "this","that","we","our","its","as","which","show","based",
        "while","both","than","can","also","have","has","many"
    }
    words = re.findall(r"\b[a-zA-Z]{4,}\b", text.lower())
    freq = {}
    for w in words:
        if w not in stopwords:
            freq[w] = freq.get(w, 0) + 1
    sorted_kw = sorted(freq.items(), key=lambda x: -x[1])[:max_keywords]
    return {
        "keywords": [kw for kw, _ in sorted_kw],
        "word_count": len(words),
        "top_keyword": sorted_kw[0][0] if sorted_kw else "",
    }

@mcp.tool
def summarize_lines(lines: List[str]) -> Dict[str, int]:
    """Return counts about a list of text lines.
    Args:
        lines: List of text lines to analyze
    """
    total = len(lines)
    nonempty = sum(1 for line in lines if line.strip())
    avg_len = sum(len(line) for line in lines) / total if total else 0
    return {
        "total_lines": total,
        "nonempty_lines": nonempty,
        "avg_line_length": round(avg_len, 1),
    }

@mcp.tool
def generate_bibliography(entries: List[str]) -> str:
    """Generate a formatted bibliography from citation strings.
    Args:
        entries: List of citation strings
    """
    lines = ["## References\n"]
    for i, entry in enumerate(entries, start=1):
        lines.append(f"[{i}] {entry.strip()}")
    return "\n".join(lines)

if __name__ == "__main__":
    mcp.run(transport="stdio")
'''
server_path.write_text(server_content, encoding="utf-8")
print("✅ Custom MCP server written to:", server_path)

## 6. Connect to all MCP servers

In [ ]:
import asyncio
import nest_asyncio
nest_asyncio.apply()

from langchain_mcp_adapters.client import MultiServerMCPClient

server_path = "/content/custom_mcp_server.py"

mcp_connections = {
    "filesystem": {
        "transport": "stdio",
        "command": "npx",
        "args": ["-y", "@modelcontextprotocol/server-filesystem", WORKDIR],
    },
    "fetch": {
        "transport": "stdio",
        "command": "npx",
        "args": ["-y", "@modelcontextprotocol/server-fetch"],
    },
    "research_tools": {
        "transport": "stdio",
        "command": "python",
        "args": [server_path],
    },
}

async def get_tools():
    client = MultiServerMCPClient(mcp_connections, tool_name_prefix=True)
    tools = await client.get_tools()
    return tools, client

tools, client = asyncio.get_event_loop().run_until_complete(get_tools())

print(f"✅ {len(tools)} tools loaded:")
for t in tools:
    print(f"  - {t.name}")

## 7. Build Gemini ReAct agent

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.prebuilt import create_react_agent
from langchain_core.messages import HumanMessage

llm = ChatGoogleGenerativeAI(
    model="gemini-1.5-flash",
    temperature=0,
    google_api_key=os.environ["GOOGLE_API_KEY"],
)

# ReAct agent: LLM decides which tools to call at each step (TAO loop)
agent = create_react_agent(llm, tools)

print("✅ Gemini ReAct agent ready")
print(f"   Model  : gemini-1.5-flash")
print(f"   Tools  : {len(tools)}")

## 8. Helper function

In [ ]:
def run_query(question: str, verbose: bool = True) -> str:
    """Run a question through the Gemini ReAct agent."""
    if verbose:
        print(f"
{chr(61)*60}")
        print(f"Q: {question}")
        print(chr(61)*60)
    result = agent.invoke({"messages": [HumanMessage(content=question)]})
    answer = result["messages"][-1].content
    if verbose:
        print(f"A: {answer}")
    return answer

## 9. Test queries

> Each query demonstrates a different MCP server being used by the agent.

In [ ]:
# Test 1 — Filesystem: list files + extract keywords from transformer paper
run_query(
    "List all the files in the workspace, then read the transformer paper "
    "and extract its keywords."
)

In [ ]:
# Test 2 — Custom tool: format a citation
run_query(
    "Format a citation for the BERT paper: "
    "authors are 'Jacob Devlin, Ming-Wei Chang, Kenton Lee, Kristina Toutanova', "
    "title is 'BERT: Pre-training of Deep Bidirectional Transformers', "
    "published in 2018 at NAACL."
)

In [ ]:
# Test 3 — Multi-step: read all papers + generate bibliography
run_query(
    "Read all three paper files in the workspace. "
    "For each one, format a proper citation. "
    "Then generate a complete bibliography with all three citations."
)

In [ ]:
# Test 4 — Fetch: Wikipedia lookup
run_query(
    "Fetch the Wikipedia page for 'transformer model machine learning' "
    "and give me 3 bullet points about it."
)

In [ ]:
# Test 5 — Full pipeline: read + summarize + write report file
run_query(
    "Read all the paper files in the workspace. "
    "Write a new file called 'research_summary.md' that contains: "
    "1) a one-sentence summary of each paper, "
    "2) their extracted keywords, "
    "3) a formatted bibliography. "
    "Confirm when done."
)

## 10. Inspect the generated report

In [ ]:
from pathlib import Path

report = Path(WORKDIR) / "research_summary.md"
if report.exists():
    print(report.read_text(encoding="utf-8"))
else:
    print("Run Test 5 first to generate the report.")